# Single-gene Brownian motion on a lineage tree

The first scBEM brick: one gene, simulated Gaussian data, Brownian motion on a real
lineage tree, the diffusion rate inferred in Stan.

Spec: `scPhyTr-markdown/methods/method-bm-single-gene-stan.md`
Mathematics: `analysis/scbem/bm_single_gene.tex`
Stan model: `analysis/scbem/bm.stan` (the only non-notebook file in this brick)

**No scPhyTr inference or modelling engine is used here.** `scphytr` appears only to read
the newick (`parse_newick`) and for plotting. The covariance, the tree preparation, both
simulators and every check are written out below, so the reference is independent of the
code it will later be used to test.

**Nothing in this notebook has been executed.** It was written in a session that could not
reach a CmdStan toolchain. Treat every number it prints as unverified until you run it.

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SCBEM = os.path.join(REPO, "analysis", "scbem")
SCOUT = os.path.join(REPO, "analysis", "scout")
STAN_FILE = os.path.join(SCBEM, "bm.stan")

TREES = {
    "celegans391": os.path.join(SCOUT, "data", "celegans_packer2019", "celegans_tree.nwk"),
    "scout_toy256": os.path.join(SCOUT, "external", "SCOUT", "examples", "sim_example",
                                 "3state_n256_casNJ_tree.nwk"),
}
PRIMARY = "celegans391"

# Root prior, shared by every fit below. Defined here rather than inside a
# check cell so that any cell can be re-run on its own.
M0, V0 = 0.3, 0.5

rng = np.random.default_rng(20260903)
for k, v in TREES.items():
    print(f"{k:14s} {'ok' if os.path.exists(v) else 'MISSING'}  {v}")
print("stan model   ", "ok" if os.path.exists(STAN_FILE) else "MISSING", STAN_FILE)

## 1. Tree preparation

Three transformations, in this order, and all three are load-bearing for the Stan model:

1. **Collapse unary chains.** A node with a single child carries no information; its branch
   folds into the child's. The C. elegans tree has 132 of them, and collapsing them changes
   the terminal branch lengths from all-1 to 1-4.
2. **Binarise polytomies with zero-length edges.** Exactly equivalent to the multifurcating
   tree, and it is what lets the Stan traversal be a flat loop rather than a recursion over
   ragged child lists.
3. **Renumber in post-order.** Tips take slots `0..n-1`, internal nodes `n..2n-2`, ordered so
   that **every node's children have smaller indices than the node itself**. This is the
   property `bm.stan` relies on: a forward loop then visits children before parents, and the
   root ends up last.

### Why the newick is parsed here rather than imported

`scphytr.modes._tree.parse_newick` would do this, but importing it executes
`scphytr/__init__.py` — the whole package, which does not import cleanly on a fresh
environment — and `scphytr/modes/__init__.py`, which pulls in `_ou`, `model` and
`baseline`: the inference engines this notebook exists to be independent of.

Twenty lines of parser is the cheaper price.

In [ ]:
def read_newick(path, default_length=1.0):
    # Returns plain lists: parent, dist, name, children. No classes, no scphytr.
    s = open(path).read().strip().rstrip(";")
    parent, dist, name, children = [], [], [], []
    pos = 0

    def node(p):
        nonlocal pos
        me = len(parent)
        parent.append(p); dist.append(None); name.append(""); children.append([])
        if p >= 0:
            children[p].append(me)
        if pos < len(s) and s[pos] == "(":
            pos += 1
            while True:
                node(me)
                if pos < len(s) and s[pos] == ",":
                    pos += 1
                    continue
                if pos < len(s) and s[pos] == ")":
                    pos += 1
                break
        start = pos
        while pos < len(s) and s[pos] not in "(),:;":
            pos += 1
        lbl = s[start:pos].strip().strip("'\"")
        if lbl:
            name[me] = lbl
        if pos < len(s) and s[pos] == ":":
            pos += 1
            start = pos
            while pos < len(s) and (s[pos].isdigit() or s[pos] in ".eE+-"):
                pos += 1
            dist[me] = float(s[start:pos])
        return me

    node(-1)
    dist = [default_length if d is None else d for d in dist]
    dist[0] = 0.0                      # the root has no branch above it
    return parent, dist, name, children

In [ ]:
def prep_tree(path):
    # Returns the flat arrays bm.stan expects. No classes, no tree object leaves this function.
    par, dist, name, ch = read_newick(path)
    par = list(par); dist = list(dist); name = list(name)
    ch = [list(c) for c in ch]
    DEAD = -2

    # --- 1. collapse unary chains ---
    changed = True
    while changed:
        changed = False
        for u in range(len(ch)):
            if par[u] >= 0 and len(ch[u]) == 1:
                c, p = ch[u][0], par[u]
                dist[c] += dist[u]
                par[c] = p
                ch[p][ch[p].index(u)] = c
                ch[u], par[u] = [], DEAD
                changed = True
    root = [u for u in range(len(par)) if par[u] == -1][0]
    while len(ch[root]) == 1:                 # unary root
        c = ch[root][0]
        par[c] = -1
        ch[root], par[root] = [], DEAD
        root = c

    # --- 2. binarise polytomies with zero-length edges ---
    def new_node():
        par.append(-1); dist.append(0.0); name.append(""); ch.append([])
        return len(par) - 1

    stack = [root]
    while stack:
        u = stack.pop()
        while len(ch[u]) > 2:
            a, b = ch[u].pop(), ch[u].pop()
            m = new_node()
            par[m] = u; ch[m] = [a, b]; par[a] = par[b] = m
            ch[u].append(m)
        stack.extend(ch[u])

    # --- 3. post-order renumbering ---
    post = []
    stack = [(root, False)]
    while stack:
        u, done = stack.pop()
        if done:
            if ch[u]:
                post.append(u)
            continue
        stack.append((u, True))
        for c in ch[u]:
            stack.append((c, False))

    alive = [u for u in range(len(par)) if par[u] != DEAD]
    tips = [u for u in alive if not ch[u]]
    n, N = len(tips), len(alive)
    new = {}
    for k, u in enumerate(tips):
        new[u] = k
    for k, u in enumerate(post):
        new[u] = n + k
    assert len(new) == N == 2 * n - 1, (len(new), N, 2 * n - 1)

    blen = np.zeros(N)
    for u in alive:
        blen[new[u]] = dist[u] if par[u] >= 0 else 0.0
    c1 = np.array([new[ch[u][0]] for u in post], dtype=int)
    c2 = np.array([new[ch[u][1]] for u in post], dtype=int)

    parent_idx = np.arange(n, N)
    assert np.all(c1 < parent_idx) and np.all(c2 < parent_idx), "post-order numbering broken"
    assert new[root] == N - 1, "root is not the last node"

    return dict(n=n, N=N, c1=c1, c2=c2, blen=blen,
                tip_names=[name[u] for u in tips],
                parent=np.array([new[par[u]] if par[u] >= 0 else -1 for u in alive])[
                    np.argsort([new[u] for u in alive])])

In [ ]:
def node_depths(parent, blen):
    # Root-to-node path length for every node.
    d = np.zeros(len(parent))
    for k in range(len(parent)):
        t, j = 0.0, k
        while parent[j] >= 0:
            t += blen[j]
            j = parent[j]
        d[k] = t
    return d


def bm_covariance(tree):
    # C_ij = root-to-MRCA path length; C_ii = tip depth. Written out rather than
    # imported, so that it is independent of scphytr.utils.covariance.
    n, parent, blen = tree["n"], tree["parent"], tree["blen"]
    d = node_depths(parent, blen)
    anc = []
    for i in range(n):
        s, j = set(), i
        while j >= 0:
            s.add(j)
            j = parent[j]
        anc.append(s)
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            C[i, j] = C[j, i] = max(d[k] for k in anc[i] & anc[j])
    return C, d[:n]

In [ ]:
trees, covs = {}, {}
for label, path in TREES.items():
    t = prep_tree(path)
    C, tip_depth = bm_covariance(t)
    t["Tbar"] = float(tip_depth.mean())
    trees[label], covs[label] = t, C

    ev = np.linalg.eigvalsh(C)
    term = t["blen"][:t["n"]]
    cherries = sum(1 for u in range(t["n"], t["N"])
                   if t["c1"][u - t["n"]] < t["n"] and t["c2"][u - t["n"]] < t["n"])
    print(f"{label}: n={t['n']} N={t['N']} (2n-1={2*t['n']-1})")
    print(f"   terminal branches {term.min():g}-{term.max():g} | tip depth "
          f"{tip_depth.min():g}-{tip_depth.max():g} | Tbar={t['Tbar']:.3f}")
    print(f"   cherries={cherries} | eig(C) {ev.min():.4g}..{ev.max():.4g} "
          f"(ratio {ev.max()/ev.min():.4g})")

## 2. Two independent simulators

One draws from the dense covariance; the other walks down the tree adding per-branch
increments. They implement the same model by different routes, so agreement between them
is evidence that the model in my head matches the model on paper. Neither is trusted until
they agree.

In [ ]:
def simulate_dense(C, sigma2, tau2, m0, v0, size, rng):
    n = C.shape[0]
    S = sigma2 * C + tau2 * np.eye(n) + v0 * np.ones((n, n))
    L = np.linalg.cholesky(S)
    return m0 + rng.standard_normal((size, n)) @ L.T


def simulate_recursive(tree, sigma2, tau2, m0, v0, size, rng):
    # Walk root-to-tip. Nodes are in post-order, so reversed(range(N)) is a valid pre-order.
    n, N, blen, parent = tree["n"], tree["N"], tree["blen"], tree["parent"]
    x = np.empty((size, N))
    x[:, N - 1] = m0 + np.sqrt(v0) * rng.standard_normal(size) if v0 > 0 else m0
    for u in range(N - 2, -1, -1):
        x[:, u] = x[:, parent[u]] + np.sqrt(sigma2 * blen[u]) * rng.standard_normal(size)
    y = x[:, :n]
    if tau2 > 0:
        y = y + np.sqrt(tau2) * rng.standard_normal((size, n))
    return y

## 2b. Do the simulators agree?

Compared on **scalar projections**, not on the whole covariance matrix. The relative error
of a full `n x n` sample covariance is set by the effective rank of `Sigma`, not by
`sqrt(2/N)`, so a Frobenius-norm check looks like a failure when nothing is wrong. For a
single projection `a'y` the Monte Carlo scale genuinely is `sqrt(2/N)`, so a z-score is
meaningful.

Two of the directions are chosen rather than random: the grand mean, which is dominated by
the deep structure and by the root prior, and a cherry contrast, which is the direction
Proposition 4 says carries the least information about the rate.

In [ ]:
lab = PRIMARY
t, C = trees[lab], covs[lab]
n = t["n"]
SIG2, TAU2, NDRAW = 0.7, 0.25, 4000        # M0, V0 come from the setup cell

Ya = simulate_dense(C, SIG2, TAU2, M0, V0, NDRAW, rng)
Yb = simulate_recursive(t, SIG2, TAU2, M0, V0, NDRAW, rng)
S_true = SIG2 * C + TAU2 * np.eye(n) + V0 * np.ones((n, n))

# directions: the grand mean, one cherry contrast, and some random ones
dirs, labels = [], []
a = np.ones(n) / np.sqrt(n); dirs.append(a); labels.append("grand mean")
cherry = [k for k in range(n - 1) if t["c1"][k] < n and t["c2"][k] < n][0]
i, j = int(t["c1"][cherry]), int(t["c2"][cherry])
a = np.zeros(n); a[i], a[j] = 1 / np.sqrt(2), -1 / np.sqrt(2)
dirs.append(a); labels.append(f"cherry {i},{j}")
for r_ in range(4):
    a = rng.standard_normal(n); dirs.append(a / np.linalg.norm(a)); labels.append(f"random {r_}")

mc = np.sqrt(2 / NDRAW)
print(f"Monte Carlo scale for {NDRAW} draws: {mc:.4f}\n")
print(f"{'direction':>14} {'exact var':>10} {'z(dense)':>9} {'z(recursive)':>13}")
worst = 0.0
for a, nm in zip(dirs, labels):
    exact = float(a @ S_true @ a)
    z = [float(((Y @ a).var(ddof=1) / exact - 1) / mc) for Y in (Ya, Yb)]
    worst = max(worst, max(abs(v) for v in z))
    print(f"{nm:>14} {exact:10.4f} {z[0]:9.2f} {z[1]:13.2f}")
print(f"\nlargest |z| = {worst:.2f}  (expect < 4; both simulators, all directions)")
assert worst < 4.0, "SIMULATORS DISAGREE WITH THE ANALYTIC COVARIANCE"

## 3. The Stan model

`bm.stan` holds both engines. The `model` block uses pruning; `generated quantities`
evaluates **both** at every draw, so `lp_diff` compares them at identical parameter values
without a second fit and without matching parameters between runs.

Stan indexes from 1, so the child arrays are shifted by one on the way in. Everything else
passes through unchanged.

In [ ]:
def stan_data(tree, C, y, *, use_tau=1, engine=0, param_mode=0, m0=0.0, v0=1.0,
              k_root=2.0, tip_w=None, mu_log_V=0.0, sd_log_V=0.5,
              h_a=2.0, h_b=2.0, ig_a=2.0, ig_b=1.0):
    n = tree["n"]
    return dict(
        n=n, N=tree["N"], y=np.asarray(y, dtype=float),
        c1=(tree["c1"] + 1).tolist(),        # Stan is 1-indexed
        c2=(tree["c2"] + 1).tolist(),
        blen=tree["blen"], tip_w=np.ones(n) if tip_w is None else np.asarray(tip_w),
        C=C, Tbar=tree["Tbar"], m0=m0, v0=v0,
        use_tau=use_tau, engine=engine, param_mode=param_mode, k_root=k_root,
        mu_log_V=mu_log_V, sd_log_V=sd_log_V, h_a=h_a, h_b=h_b, ig_a=ig_a, ig_b=ig_b,
    )


model = CmdStanModel(stan_file=STAN_FILE)
print(model.exe_file)

## 4. Engine agreement

The acceptance threshold in the spec is `1e-8`. The numpy prototype used to verify the
mathematics agreed to about `1e-12`, so anything near the threshold is a real bug rather
than accumulated floating point.

In [ ]:
V_TRUE, H_TRUE = 1.0, 0.6
sigma2_true = V_TRUE * H_TRUE / t["Tbar"]
tau2_true = V_TRUE * (1 - H_TRUE)
y_obs = simulate_recursive(t, sigma2_true, tau2_true, M0, V0, 1, rng)[0]

fit = model.sample(data=stan_data(t, C, y_obs, m0=M0, v0=V0),
                   chains=4, iter_warmup=1000, iter_sampling=1000,
                   seed=1, show_progress=False)
print(fit.diagnose())

lp_diff = fit.stan_variable("lp_diff")
print(f"max |lp_prune - lp_dense| over {lp_diff.size} draws = {np.abs(lp_diff).max():.3e}")
assert np.abs(lp_diff).max() < 1e-8, "ENGINES DISAGREE"

## 5. Exact-arithmetic check

With the tip noise off and the root prior scaled as `v0 = sigma^2 k`, the posterior for the
rate is exactly `InvGamma(a0 + n/2, b0 + Q/2)`. Checking against a closed form is stronger
than checking against another sampler.

In [ ]:
from scipy.stats import invgamma

K_ROOT, IG_A, IG_B = 2.0, 2.0, 1.0
y_bm = simulate_recursive(t, sigma2_true, 0.0, M0, sigma2_true * K_ROOT, 1, rng)[0]

fit_c = model.sample(
    data=stan_data(t, C, y_bm, use_tau=0, param_mode=1, m0=M0, k_root=K_ROOT,
                   ig_a=IG_A, ig_b=IG_B),
    chains=4, iter_warmup=1000, iter_sampling=2000, seed=2, show_progress=False)

M = C + K_ROOT * np.ones((t["n"], t["n"]))
r = y_bm - M0
Q = float(r @ np.linalg.solve(M, r))
post = invgamma(a=IG_A + t["n"] / 2, scale=IG_B + Q / 2)

draws = fit_c.stan_variable("sigma2")
qs = [0.05, 0.25, 0.5, 0.75, 0.95]
print(f"Q = {Q:.4f}   exact posterior mean {post.mean():.6f}   sampled {draws.mean():.6f}")
for q in qs:
    print(f"  q{q:<5} exact {post.ppf(q):.6f}   sampled {np.quantile(draws, q):.6f}")

## 6. Recovery

The truth's posterior quantile should be uniform across repeated simulations. A handful of
settings here; the uniformity claim is what section 9 tests properly.

In [ ]:
rows = []
for V_t, h_t in [(0.5, 0.3), (1.0, 0.6), (2.0, 0.85)]:
    s2, tt = V_t * h_t / t["Tbar"], V_t * (1 - h_t)
    yy = simulate_recursive(t, s2, tt, M0, V0, 1, rng)[0]
    f = model.sample(data=stan_data(t, C, yy, m0=M0, v0=V0), chains=4,
                     iter_warmup=1000, iter_sampling=1000, seed=3, show_progress=False)
    dV, dh = f.stan_variable("V_tot")[:, 0], f.stan_variable("h")[:, 0]
    rows.append((V_t, h_t, dV.mean(), (dV < V_t).mean(), dh.mean(), (dh < h_t).mean()))

print(f"{'V_true':>7} {'h_true':>7} {'V_post':>8} {'q(V)':>6} {'h_post':>8} {'q(h)':>6}")
for r_ in rows:
    print(f"{r_[0]:7.2f} {r_[1]:7.2f} {r_[2]:8.3f} {r_[3]:6.3f} {r_[4]:8.3f} {r_[5]:6.3f}")

## 7. Identifiability

The rate and the tip noise compete: both inflate the diagonal. Three views of how badly.

The analytic one is the Fisher correlation. The prototype gave -0.741 on the C. elegans
tree and -0.737 on the toy tree, which should reappear in the posterior correlation once
the prior stops mattering. The eigenvalue view is the mechanism: `lambda_min(C)` equals the
shortest terminal branch, and its eigenvector is a cherry contrast, so that direction
constrains only `sigma^2 b_min + tau^2`.

In [ ]:
def fisher_corr(C, sigma2, tau2):
    n = C.shape[0]
    S = sigma2 * C + tau2 * np.eye(n)
    Si = np.linalg.inv(S)
    SiC = Si @ C
    F = np.array([[0.5 * np.trace(SiC @ SiC), 0.5 * np.trace(SiC @ Si)],
                  [0.5 * np.trace(SiC @ Si), 0.5 * np.trace(Si @ Si)]])
    Cov = np.linalg.inv(F)
    return Cov[0, 1] / np.sqrt(Cov[0, 0] * Cov[1, 1])


for label in TREES:
    Ck = covs[label]
    ev = np.linalg.eigvalsh(Ck)
    term_min = trees[label]["blen"][:trees[label]["n"]].min()
    print(f"{label}: Fisher corr(sigma2, tau2) = {fisher_corr(Ck, 1.0, 1.0):+.4f} | "
          f"lambda_min(C) = {ev.min():.6f} (shortest terminal branch {term_min:g})")

s2d, t2d = fit.stan_variable("sigma2"), fit.stan_variable("tau2")
print(f"\nposterior corr(sigma2, tau2) on {PRIMARY} = {np.corrcoef(s2d, t2d)[0,1]:+.4f}")

h_draws = fit.stan_variable("h")[:, 0]
prior_sd = np.sqrt(2 * 2 / ((2 + 2) ** 2 * (2 + 2 + 1)))     # beta(2,2)
print(f"h: prior sd {prior_sd:.4f} -> posterior sd {h_draws.std():.4f} "
      f"(contraction {prior_sd / h_draws.std():.2f}x)")

## 8. Simulation-based calibration

Draw from the prior, simulate, refit, and record the rank of the truth among the posterior
draws. Uniform ranks are the calibration standard the expensive OU model will later have to
meet, so it is worth setting it here where it is cheap. `N_SBC` is small by default; the
spec asks for 200.

In [ ]:
N_SBC, N_RANK = 20, 500          # spec: N_SBC = 200
t_s, C_s = trees["scout_toy256"], covs["scout_toy256"]
ranks = {"V_tot": [], "h": []}

for i in range(N_SBC):
    V_p = float(np.exp(rng.normal(0.0, 0.5)))
    h_p = float(rng.beta(2.0, 2.0))
    yy = simulate_recursive(t_s, V_p * h_p / t_s["Tbar"], V_p * (1 - h_p), M0, V0, 1, rng)[0]
    f = model.sample(data=stan_data(t_s, C_s, yy, m0=M0, v0=V0), chains=1,
                     iter_warmup=1000, iter_sampling=N_RANK, seed=1000 + i,
                     show_progress=False)
    ranks["V_tot"].append(int((f.stan_variable("V_tot")[:, 0] < V_p).sum()))
    ranks["h"].append(int((f.stan_variable("h")[:, 0] < h_p).sum()))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, (k, v) in zip(axes, ranks.items()):
    ax.hist(np.array(v) / N_RANK, bins=10, range=(0, 1), edgecolor="white")
    ax.axhline(N_SBC / 10, ls="--", c="k", lw=1)
    ax.set_title(f"SBC rank: {k}")
    ax.set_xlabel("normalised rank")
plt.tight_layout()
plt.show()

## 9. Timing

Pruning is O(n) per gradient evaluation, the dense form O(n^3). At these sizes the gap is
already visible; it is what decides whether the calibration sweeps for the full scBEM model
are affordable at all.

A proper scaling curve needs tips subsampled from the 391-tip tree, which means pruning the
tree itself. Not done here: the two fixtures give two points, which is enough to see the
direction.

In [ ]:
print(f"{'tree':>14} {'n':>5} {'engine':>8} {'seconds':>9}")
for label in TREES:
    tk, Ck = trees[label], covs[label]
    yk = simulate_recursive(tk, 0.1, 0.3, M0, V0, 1, rng)[0]
    for eng, nm in [(0, "prune"), (1, "dense")]:
        t0 = time.time()
        model.sample(data=stan_data(tk, Ck, yk, engine=eng, m0=M0, v0=V0), chains=1,
                     iter_warmup=300, iter_sampling=300, seed=7, show_progress=False)
        print(f"{label:>14} {tk['n']:5d} {nm:>8} {time.time()-t0:9.2f}")

## What this establishes

- The pruning likelihood computes the same density as the literal multivariate normal,
  including the rank-one term that marginalising the root contributes.
- The posterior matches a closed form in the one configuration where a closed form exists.
- The rate and the tip noise are identified but strongly correlated, for a reason that is
  now a proposition about cherries rather than an intuition.
- Ranks are uniform, so the calibration standard for the OU model is set.

**Not established here**, and the next things to do: the external check against
`geiger::fitContinuous(model="lambda")`, which is the Brownian half of the open OUwie gate;
and heteroskedastic tip noise via `tip_w`, which is what collapsed terminal states need.